# 우수모델 성능 결과

In [16]:
# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score, accuracy_score
)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")
import os


# ============================================================
# 1. 설정값
# ============================================================

TRAIN_PATH   = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH    = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'
FEATURE_PATH = r'13번.피처셀렉션\M19_도매_소매업\lasso_features_top55--45.csv'

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
THRESHOLD    = 0.44
YEAR_COL     = "회계년도"
ID_COLS      = ["회사명", "사업자등록번호", "회계년도"]

FOLD_VAL_YEARS = [2016, 2017, 2018, 2019, 2020, 2021]
TRAIN_START    = 2012

SAVE_DIR = r'15번. 우수모델 데이터'

# ============================================================
# 2. 데이터 로드
# ============================================================

train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)

y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]

feat_df      = pd.read_csv(FEATURE_PATH)
col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]

pos_weight = (y_train_full == 0).sum() / (y_train_full == 1).sum()

print("=" * 65)
print(f"Train shape  : {train_full.shape}")
print(f"Test  shape  : {test.shape}")
print(f"피처 수       : {len(use_features)}개")
print(f"pos_weight   : {pos_weight:.4f}")
print(f"threshold    : {THRESHOLD}")
print("=" * 65)


# ============================================================
# 3. 모델 정의
# ============================================================

def make_model():
    return XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
        scale_pos_weight=pos_weight
    )


# ============================================================
# 4. 평가 헬퍼
# ============================================================

def calc_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "ROC_AUC"   : roc_auc_score(y_true, y_prob),
        "PR_AUC"    : average_precision_score(y_true, y_prob),
        "Accuracy"  : accuracy_score(y_true, y_pred),
    }


# ============================================================
# 5. Expanding Window CV
# ============================================================

print("\nExpanding Window CV")
print("=" * 65)

cv_rows      = []
val_prob_all = []   # 전체 val 확률 수집 (분포용)

for val_year in FOLD_VAL_YEARS:
    train_idx = train_full.index[
        (train_full[YEAR_COL] >= TRAIN_START) & (train_full[YEAR_COL] < val_year)
    ]
    val_idx = train_full.index[train_full[YEAR_COL] == val_year]

    if len(train_idx) == 0 or len(val_idx) == 0:
        print(f"  [경고] val_year={val_year} 데이터 없음 → skip")
        continue

    X_fold_train = train_full.loc[train_idx, use_features]
    y_fold_train = train_full.loc[train_idx, TARGET_COL]
    X_fold_val   = train_full.loc[val_idx,   use_features]
    y_fold_val   = train_full.loc[val_idx,   TARGET_COL]

    imputer      = SimpleImputer(strategy="median")
    X_fold_train = pd.DataFrame(imputer.fit_transform(X_fold_train), columns=use_features)
    X_fold_val   = pd.DataFrame(imputer.transform(X_fold_val),       columns=use_features)

    model = make_model()
    model.fit(X_fold_train, y_fold_train)
    y_prob_val = model.predict_proba(X_fold_val)[:, 1]

    m = calc_metrics(y_fold_val, y_prob_val)
    print(f"  Val {val_year} | F1={m['F1']:.4f} Recall={m['Recall']:.4f} "
          f"Precision={m['Precision']:.4f} ROC_AUC={m['ROC_AUC']:.4f} PR_AUC={m['PR_AUC']:.4f}")

    cv_rows.append({"Val_Year": val_year, **m})
    val_prob_all.append(pd.DataFrame({
        "Val_Year"   : val_year,
        "y_true"     : y_fold_val.values,
        "y_prob"     : y_prob_val,
        "y_pred"     : (y_prob_val >= THRESHOLD).astype(int),
    }))

cv_df = pd.DataFrame(cv_rows).round(4)
cv_mean = cv_df[["F1","Recall","Precision","ROC_AUC","PR_AUC","Accuracy"]].mean()

print(f"\n  {'─'*55}")
print(f"  CV 평균 | F1={cv_mean['F1']:.4f} Recall={cv_mean['Recall']:.4f} "
      f"Precision={cv_mean['Precision']:.4f} ROC_AUC={cv_mean['ROC_AUC']:.4f} "
      f"PR_AUC={cv_mean['PR_AUC']:.4f}")


# ============================================================
# 6. Test 평가 (전체 train 재학습)
# ============================================================

print(f"\n{'='*65}")
print("Test 평가")
print("=" * 65)

imputer_final = SimpleImputer(strategy="median")
X_train_all   = pd.DataFrame(
    imputer_final.fit_transform(train_full[use_features]), columns=use_features
)
X_test_imp    = pd.DataFrame(
    imputer_final.transform(test[use_features]), columns=use_features
)

final_model = make_model()
final_model.fit(X_train_all, y_train_full)
y_prob_test = final_model.predict_proba(X_test_imp)[:, 1]
y_pred_test = (y_prob_test >= THRESHOLD).astype(int)

test_metrics = calc_metrics(y_test, y_prob_test)
print(f"  Test    | F1={test_metrics['F1']:.4f} Recall={test_metrics['Recall']:.4f} "
      f"Precision={test_metrics['Precision']:.4f} ROC_AUC={test_metrics['ROC_AUC']:.4f} "
      f"PR_AUC={test_metrics['PR_AUC']:.4f}")

print(f"\n  [Gap = Test - CV평균]")
for col in ["F1","Recall","Precision","ROC_AUC","PR_AUC","Accuracy"]:
    gap = test_metrics[col] - cv_mean[col]
    print(f"    {col:<12} : {gap:+.4f}")


# ============================================================
# 7. 확률 분포 시각화 저장
# ============================================================

import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ── 폰트/스타일 설정 ─────────────────────────────────────
plt.rcParams.update({
    "font.family"       : "DejaVu Sans",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.color"        : "#E5E5E5",
    "grid.linewidth"    : 0.7,
    "axes.facecolor"    : "#FAFAFA",
    "figure.facecolor"  : "white",
})

COLOR_NEG   = "#2F6EBA"   # Normal(0) — 파랑
COLOR_POS   = "#D94F3D"   # Distress(1) — 빨강
COLOR_THR   = "#F5A623"   # threshold — 주황
ALPHA_HIST  = 0.72
BINS        = 45

val_prob_df = pd.concat(val_prob_all, ignore_index=True)

# ── 레이아웃: 2열 메인 + 하단 메트릭 테이블 ──────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(
    2, 2,
    height_ratios=[3.2, 1],
    hspace=0.42, wspace=0.32,
    left=0.07, right=0.97, top=0.91, bottom=0.05
)

ax_cv   = fig.add_subplot(gs[0, 0])
ax_test = fig.add_subplot(gs[0, 1])
ax_tbl  = fig.add_subplot(gs[1, :])
ax_tbl.axis("off")

# ── 공통 플롯 함수 ────────────────────────────────────────
def plot_dist(ax, y_true, y_prob, title, n_total):
    arr0 = y_prob[np.array(y_true) == 0]
    arr1 = y_prob[np.array(y_true) == 1]

    counts0, edges0 = np.histogram(arr0, bins=BINS, range=(0, 1))
    counts1, edges1 = np.histogram(arr1, bins=BINS, range=(0, 1))

    ax.bar(edges0[:-1], counts0, width=np.diff(edges0),
           align="edge", color=COLOR_NEG, alpha=ALPHA_HIST, label="Normal (0)", zorder=3)
    ax.bar(edges1[:-1], counts1, width=np.diff(edges1),
           align="edge", color=COLOR_POS, alpha=ALPHA_HIST, label="Distress (1)", zorder=3)

    # threshold 수직선
    ax.axvline(THRESHOLD, color=COLOR_THR, linestyle="--",
               linewidth=1.8, zorder=5, label=f"Threshold = {THRESHOLD}")

    # 음영 — threshold 우측 위험 영역
    ax.axvspan(THRESHOLD, 1.0, alpha=0.06, color=COLOR_POS, zorder=2)

    # 통계 annotation
    n0, n1 = len(arr0), len(arr1)
    ir = n1 / n0 if n0 > 0 else float("nan")
    above_thr = (y_prob >= THRESHOLD).sum()
    stats_txt = (
        f"N={n_total:,}  |  Normal={n0:,}  Distress={n1:,}\n"
        f"Imbalance ratio = {ir:.3f}  |  Predicted Positive = {above_thr:,}"
    )
    ax.text(0.98, 0.97, stats_txt,
            transform=ax.transAxes, fontsize=8.2,
            va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#CCCCCC", alpha=0.85))

    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Predicted Probability", fontsize=9.5)
    ax.set_ylabel("Count", fontsize=9.5)
    ax.set_xlim(0, 1)
    ax.tick_params(labelsize=8.5)

    legend_elems = [
        Patch(facecolor=COLOR_NEG, alpha=ALPHA_HIST, label="Normal (0)"),
        Patch(facecolor=COLOR_POS, alpha=ALPHA_HIST, label="Distress (1)"),
        Line2D([0], [0], color=COLOR_THR, linestyle="--", linewidth=1.8,
               label=f"Threshold = {THRESHOLD}"),
    ]
    ax.legend(handles=legend_elems, fontsize=8.5, framealpha=0.9,
              loc="upper left", edgecolor="#CCCCCC")

plot_dist(ax_cv,
          val_prob_df["y_true"].values,
          val_prob_df["y_prob"].values,
          "Expanding Window CV — Predicted Probability Distribution",
          n_total=len(val_prob_df))

plot_dist(ax_test,
          y_test.values,
          y_prob_test,
          "Hold-out Test — Predicted Probability Distribution",
          n_total=len(y_test))

# ── 하단 메트릭 테이블 ────────────────────────────────────
metrics_order = ["F1", "Recall", "Precision", "ROC_AUC", "PR_AUC", "Accuracy"]
col_labels    = ["Split"] + metrics_order

cv_row   = ["CV Mean"] + [f"{float(cv_mean[m]):.4f}"   for m in metrics_order]
test_row = ["Test"]    + [f"{test_metrics[m]:.4f}"     for m in metrics_order]
gap_row  = ["Gap (Test − CV)"] + [
    f"{test_metrics[m] - float(cv_mean[m]):+.4f}" for m in metrics_order
]

table_data = [cv_row, test_row, gap_row]

tbl = ax_tbl.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.7)

# 헤더 스타일
for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor("#2F4F7F")
    tbl[(0, j)].set_text_props(color="white", fontweight="bold")

# 행 배경
row_colors = ["#EEF3FA", "#FAFAFA", "#FFF4EE"]
for i, rc in enumerate(row_colors, start=1):
    for j in range(len(col_labels)):
        tbl[(i, j)].set_facecolor(rc)

# Gap 행 — 음수(빨강) / 양수(초록) 색상
for j, m in enumerate(metrics_order, start=1):
    gap_val = test_metrics[m] - float(cv_mean[m])
    color   = "#C0392B" if gap_val < -0.02 else ("#27AE60" if gap_val > 0.02 else "#555555")
    tbl[(3, j)].set_text_props(color=color, fontweight="bold")

ax_tbl.set_title("Performance Summary", fontsize=10,
                 fontweight="bold", pad=6, loc="left")

# ── 전체 제목 ─────────────────────────────────────────────
fig.suptitle(
    "XGBoost  │  Class Weight (scale_pos_weight)  │  M19 도매·소매업",
    fontsize=13.5, fontweight="bold", y=0.975
)

plt.savefig(
    os.path.join(SAVE_DIR, "XGB_CW_prob_distribution.png"),
    dpi=180, bbox_inches="tight"
)
plt.close()
print(f"\n  확률 분포 이미지 저장 완료")



# ============================================================
# 8. Test 전체 행 예측 결과 저장
# ============================================================

# ID 컬럼 붙이기
test_id_cols = [c for c in ID_COLS if c in test.columns]
test_pred_df = test[test_id_cols].copy().reset_index(drop=True)
test_pred_df["y_true"]       = y_test.values
test_pred_df["y_prob"]       = y_prob_test.round(4)
test_pred_df["y_pred"]       = y_pred_test
test_pred_df["correct"]      = (test_pred_df["y_true"] == test_pred_df["y_pred"]).astype(int)
test_pred_df["error_type"]   = "TN"
test_pred_df.loc[(test_pred_df["y_true"]==1) & (test_pred_df["y_pred"]==1), "error_type"] = "TP"
test_pred_df.loc[(test_pred_df["y_true"]==1) & (test_pred_df["y_pred"]==0), "error_type"] = "FN"
test_pred_df.loc[(test_pred_df["y_true"]==0) & (test_pred_df["y_pred"]==1), "error_type"] = "FP"

print(f"\n  [Test 예측 분포]")
print(test_pred_df["error_type"].value_counts().to_string())


# ============================================================
# 9. 저장
# ============================================================

# CV fold별 성능
import os
SAVE_DIR = r'15번. 우수모델 데이터'
os.makedirs(SAVE_DIR, exist_ok=True)

cv_df.to_csv(os.path.join(SAVE_DIR, "XGB_CW_CV_results.csv"), index=False, encoding="utf-8-sig")

# CV평균 + Test + Gap 요약
summary = {"Method": "ClassWeight", "FeatureFile": FEATURE_PATH.split("\\")[-1],
           "N_Features": len(use_features), "threshold": THRESHOLD}
for col in ["F1","Recall","Precision","ROC_AUC","PR_AUC","Accuracy"]:
    summary[f"CV_Val_{col}"] = round(float(cv_mean[col]), 4)
    summary[f"Test_{col}"]   = round(test_metrics[col], 4)
    summary[f"Gap_{col}"]    = round(test_metrics[col] - float(cv_mean[col]), 4)
pd.DataFrame([summary]).to_csv(os.path.join(SAVE_DIR, "XGB_CW_summary.csv"), index=False, encoding="utf-8-sig")

# Test 전체 행 예측 결과
test_pred_df.to_csv(os.path.join(SAVE_DIR, "XGB_CW_test_predictions.csv"), index=False, encoding="utf-8-sig")

# CV val 전체 확률 분포 raw
val_prob_df.to_csv(os.path.join(SAVE_DIR, "XGB_CW_val_prob_distribution.csv"), index=False, encoding="utf-8-sig")

# ============================================================
# 10. 2021~2024 전체 PD 데이터 생성 및 저장
# ============================================================

PD_YEARS = [2021, 2022, 2023, 2024]
PD_SAVE_DIR = r'15번. 우수모델 데이터'
os.makedirs(PD_SAVE_DIR, exist_ok=True)

# ── 전체 데이터 (train + test) 합치기 ────────────────────
all_data = pd.concat([train_full, test], ignore_index=True)
all_data_years = all_data[all_data[YEAR_COL].isin(PD_YEARS)].copy()

# ── imputation & 예측 ─────────────────────────────────────
# imputer_final, final_model은 섹션 6에서 이미 학습된 상태
X_all_imp = pd.DataFrame(
    imputer_final.transform(all_data_years[use_features]),
    columns=use_features,
    index=all_data_years.index
)

all_data_years["y_prob"] = final_model.predict_proba(X_all_imp)[:, 1].round(4)
all_data_years["y_pred"] = (all_data_years["y_prob"] >= THRESHOLD).astype(int)
all_data_years["y_true"] = all_data_years[TARGET_COL]
all_data_years["correct"] = (
    all_data_years["y_true"] == all_data_years["y_pred"]
).astype(int)

# error_type 분류
all_data_years["error_type"] = "TN"
all_data_years.loc[
    (all_data_years["y_true"]==1) & (all_data_years["y_pred"]==1), "error_type"
] = "TP"
all_data_years.loc[
    (all_data_years["y_true"]==1) & (all_data_years["y_pred"]==0), "error_type"
] = "FN"
all_data_years.loc[
    (all_data_years["y_true"]==0) & (all_data_years["y_pred"]==1), "error_type"
] = "FP"

# ── 컬럼 정리 ─────────────────────────────────────────────
id_cols_available = [c for c in ID_COLS if c in all_data_years.columns]
pd_df = all_data_years[id_cols_available + [
    "y_true", "y_prob", "y_pred", "correct", "error_type"
]].sort_values(
    [id_cols_available[1], YEAR_COL]   # 사업자등록번호, 회계년도 순 정렬
).reset_index(drop=True)



# ── 저장 ──────────────────────────────────────────────────
pd_save_path = os.path.join(PD_SAVE_DIR, "2021_2024_PD_데이터.csv")
pd_df.to_csv(pd_save_path, index=False, encoding="utf-8-sig")

print(f"\n{'='*65}")
print(f"2021~2024 PD 데이터 저장 완료")
print(f"  → {pd_save_path}  ({len(pd_df)}행)")
print(f"\n  [연도별 행 수]")
print(pd_df[YEAR_COL].value_counts().sort_index().to_string())
print(f"\n  [error_type 분포]")
print(pd_df["error_type"].value_counts().to_string())
print("=" * 65)





# ============================================================
# 11. 2014~2024 전체 PD 데이터 생성 및 저장
# ============================================================

PD_YEARS = [2014,2015,2016,2017,2018,2019,2020,2021, 2022, 2023, 2024]
PD_SAVE_DIR = r'15번. 우수모델 데이터'
os.makedirs(PD_SAVE_DIR, exist_ok=True)

# ── 전체 데이터 (train + test) 합치기 ────────────────────
all_data = pd.concat([train_full, test], ignore_index=True)
all_data_years = all_data[all_data[YEAR_COL].isin(PD_YEARS)].copy()

# ── imputation & 예측 ─────────────────────────────────────
# imputer_final, final_model은 섹션 6에서 이미 학습된 상태
X_all_imp = pd.DataFrame(
    imputer_final.transform(all_data_years[use_features]),
    columns=use_features,
    index=all_data_years.index
)

all_data_years["y_prob"] = final_model.predict_proba(X_all_imp)[:, 1].round(4)
all_data_years["y_pred"] = (all_data_years["y_prob"] >= THRESHOLD).astype(int)
all_data_years["y_true"] = all_data_years[TARGET_COL]
all_data_years["correct"] = (
    all_data_years["y_true"] == all_data_years["y_pred"]
).astype(int)

# error_type 분류
all_data_years["error_type"] = "TN"
all_data_years.loc[
    (all_data_years["y_true"]==1) & (all_data_years["y_pred"]==1), "error_type"
] = "TP"
all_data_years.loc[
    (all_data_years["y_true"]==1) & (all_data_years["y_pred"]==0), "error_type"
] = "FN"
all_data_years.loc[
    (all_data_years["y_true"]==0) & (all_data_years["y_pred"]==1), "error_type"
] = "FP"

# ── 컬럼 정리 ─────────────────────────────────────────────
id_cols_available = [c for c in ID_COLS if c in all_data_years.columns]
pd_df = all_data_years[id_cols_available + [
    "y_true", "y_prob", "y_pred", "correct", "error_type"
]].sort_values(
    [id_cols_available[1], YEAR_COL]   # 사업자등록번호, 회계년도 순 정렬
).reset_index(drop=True)



# ── 저장 ──────────────────────────────────────────────────
pd_save_path = os.path.join(PD_SAVE_DIR, "2014_2024_PD_데이터.csv")
pd_df.to_csv(pd_save_path, index=False, encoding="utf-8-sig")

print(f"\n{'='*65}")
print(f"2014~2024 PD 데이터 저장 완료")
print(f"  → {pd_save_path}  ({len(pd_df)}행)")
print(f"\n  [연도별 행 수]")
print(pd_df[YEAR_COL].value_counts().sort_index().to_string())
print(f"\n  [error_type 분포]")
print(pd_df["error_type"].value_counts().to_string())
print("=" * 65)



Train shape  : (28111, 256)
Test  shape  : (11797, 256)
피처 수       : 45개
pos_weight   : 25.6455
threshold    : 0.44

Expanding Window CV
  Val 2016 | F1=0.4153 Recall=0.6622 Precision=0.3025 ROC_AUC=0.9536 PR_AUC=0.4580
  Val 2017 | F1=0.4353 Recall=0.5286 Precision=0.3700 ROC_AUC=0.9568 PR_AUC=0.3959
  Val 2018 | F1=0.4589 Recall=0.7444 Precision=0.3317 ROC_AUC=0.9587 PR_AUC=0.3685
  Val 2019 | F1=0.4099 Recall=0.7830 Precision=0.2776 ROC_AUC=0.9530 PR_AUC=0.4657
  Val 2020 | F1=0.3620 Recall=0.5900 Precision=0.2611 ROC_AUC=0.9479 PR_AUC=0.3244
  Val 2021 | F1=0.4791 Recall=0.8651 Precision=0.3313 ROC_AUC=0.9717 PR_AUC=0.4840

  ───────────────────────────────────────────────────────
  CV 평균 | F1=0.4267 Recall=0.6955 Precision=0.3124 ROC_AUC=0.9569 PR_AUC=0.4161

Test 평가
  Test    | F1=0.4203 Recall=0.9119 Precision=0.2731 ROC_AUC=0.9597 PR_AUC=0.4525

  [Gap = Test - CV평균]
    F1           : -0.0064
    Recall       : +0.2163
    Precision    : -0.0393
    ROC_AUC      : +0.0027
    